In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import xsdba
import matplotlib.pyplot as plt
import seaborn as sns
import cmocean

plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=10)

In [2]:
# Root folder of the data (edit this path, same as in multivariate_hindcast.py)
p = Path('wave-climate-ITbuoys')

dir_bc =  p / 'bias-adj-Hs_Tp-dOTC'                     # bias-adjusted data (dOTC)
dir_reg = dir_bc / 'hindcast_MeteOcean-UniGe-ww3-10km/'  # reference: hindcast corrected by multivariate_hindcast.py

dir_pics = Path('pics')

dproj = 'projections_MeteOcean-UniGe-ww3-10km'
# input:  p / 'raw' / dproj / 'baseline-1970-2005' and 'rcp85-2006-2100'
# output: dir_bc / dproj / 'baseline-1970-2005' and 'rcp85-2006-2100'
for sub in ['baseline-1970-2005', 'rcp85-2006-2100']:
    (dir_bc / dproj / sub).mkdir(parents=True, exist_ok=True)

models = {
    1:["CLMcom-CCCma-CanESM2", "CCLM4-CanESM2"],
    2:["CLMcom-MIROC-MIROC5", "CCLM4-MIROC5"],
    3:["SMHI-MPI-M-MPI-ESM-LR", "RCA4-MPI-ESM-LR"],
    4:["SMHI-NCC-NorESM1-M", "RCA4-NorESM1-M"],
    5:["SMHI-CNRM-CERFACS-CNRM-CM5", "RCA4-CNRM-CM5"],
    6:["SMHI-IPSL-IPSL-CM5A-MR", "RCA4-IPSL-CM5A-MR"],
    7:["SMHI-MOHC-HadGEM2-ES", "RCA4-HadGEM2-ES"],
    8:["SMHI-ICHEC-EC-EARTH", "RCA4-EC-EARTH"],
    9:["DMI-ICHEC-EC-EARTH", "HIRHAM5-EC-EARTH"],
    10:["DMI-NCC-NorESM1-M", "HIRHAM5-NorESM1-M"],
    11:["DMI-MOHC-HadGEM2-ES", "HIRHAM5-HadGEM2-ES"],
    12:["DMI-MPI-M-MPI-ESM-LR", "HIRHAM5-MPI-ESM-LR"],
    13:["DMI-CNRM-CERFACS-CNRM-CM5", "HIRHAM5-CNRM-CM5"],
    14:["DMI-IPSL-IPSL-CM5A-MR", "HIRHAM5-IPSL-CM5A-MR"],
    15:["CLMcom-ETH-ICHEC-EC-EARTH", "COSMO-crCLIM1-EC-EARTH"],
    16:["CLMcom-ETH-NCC-NorESM1-M", "COSMO-crCLIM1-NorESM1-M"],
    17:["CLMcom-ETH-MOHC-HadGEM2-ES", "COSMO-crCLIM1-HadGEM2-ES"],
    18:["KNMI-CNRM-CERFACS-CNRM-CM5", "RACMO22E-CNRM-CM5"],
    19:["KNMI-ICHEC-EC-EARTH", "RACMO22E-EC-EARTH"],
    20:["KNMI-IPSL-IPSL-CM5A-MR", "RACMO22E-IPSL-CM5A-MR"],
    21:["KNMI-MOHC-HadGEM2-ES", "RACMO22E-HadGEM2-ES"]
    }

In [ ]:
# Stations to process (they replace the file coordinateDR.xlsx).
# Keep only the stations whose files are available.
stations = ['spezia', 'capomele', 'alghero', 'crotone', 'cagliari', 'mazara', 'monopoli', 'civitavecchia', 'ortona', 'ancona', 'cetraro', 'cesenatico', 'ponza', 'palermo', 'catania']

In [ ]:
for pp, nn in enumerate(stations):
    
    nn = nn.upper()
    print(nn)
    reg = pd.read_csv(dir_reg / f'{nn}_ww3-10km_hindcast.txt', index_col=0, parse_dates=True)
    reg.dropna(inplace=True)
    reg = reg.rename(columns={'hs_ba':'hs', 'tp_ba':'tp'})

    for mm in range(1, 22):
        mod = models[mm]
        df_modh = pd.read_csv(p / 'raw' / dproj / 'baseline-1970-2005' / f'{nn}_ww3-10km_baseline_{mod[1]}.txt', index_col=0, parse_dates=True)
        df_modf = pd.read_csv(p / 'raw' / dproj / 'rcp85-2006-2100' / f'{nn}_ww3-10km_rcp85_{mod[1]}.txt', index_col=0, parse_dates=True)

        df = pd.concat([reg[['hs', 'tp']], df_modh[['hs', 'tp']]], keys=["ww3-10km", mod[1]], axis=1)
        df = df.dropna(axis=0)

        ref = df['ww3-10km'].to_xarray()
        ref.hs.attrs['units'] = 'm'
        ref.tp.attrs['units'] = 's'

        hist = df[mod[1]].to_xarray()
        hist.hs.attrs['units'] = 'm'
        hist.tp.attrs['units'] = 's'

        sim_hist = df_modh[['hs', 'tp']].to_xarray()
        sim_hist.hs.attrs['units'] = 'm'
        sim_hist.tp.attrs['units'] = 's'

        sim_fut = df_modf[['hs', 'tp']].to_xarray()
        sim_fut.hs.attrs['units'] = 'm'
        sim_fut.tp.attrs['units'] = 's'

        ref = xsdba.processing.stack_variables(ref)
        hist = xsdba.processing.stack_variables(hist)
        sim_hist = xsdba.processing.stack_variables(sim_hist)
        sim_fut = xsdba.processing.stack_variables(sim_fut)

        #OTC
        mydOTC = xsdba.dOTC
        mydOTC._allow_diff_time_sizes = True
        # xsdba.adjustment.dOTC.adjust(
        sim_hist_ba = mydOTC.adjust(sim=sim_hist, ref=ref, hist=hist, group="time",
                                    kind={"hs": "+", "tp":"+"},  
                                    )

        df_hist_ba = xsdba.unstack_variables(sim_hist_ba).to_dataframe()
        df_hist_ba = df_hist_ba.rename(columns={'hs':'hs_ba', 'tp':'tp_ba'})
        df_hist_ba = df_hist_ba.round({'hs_ba': 2, 'tp_ba':2})
        df_hist_ba.index.names = ['time']
        df_hist_ba.to_csv(dir_bc / dproj / 'baseline-1970-2005' / f'{nn}_ww3-10km_baseline_{mod[1]}.txt')

        sim_fut_ba = mydOTC.adjust(sim=sim_fut, ref=ref, hist=hist, group="time",
                                    kind={"hs": "+", "tp":"+"},  
                                    )

        df_fut_ba = xsdba.unstack_variables(sim_fut_ba).to_dataframe()
        df_fut_ba = df_fut_ba.rename(columns={'hs':'hs_ba', 'tp':'tp_ba'})
        df_fut_ba = df_fut_ba.round({'hs_ba': 2, 'tp_ba':2})
        df_fut_ba.index.names = ['time']
        df_fut_ba.to_csv(dir_bc / dproj / 'rcp85-2006-2100' / f'{nn}_ww3-10km_rcp85_{mod[1]}.txt')
